# 1. Import Dependencies

In [1]:
import os
import gym
import numpy as np
import pandas as pd
import random
import time
from matplotlib import pyplot as plt
from gym.spaces import Discrete, Box, Dict, Tuple, MultiBinary, MultiDiscrete 
import torch.nn as nn
from typing import Optional, List
import gym.spaces as spaces

# Git Push Pull testing

# State settings

In [8]:
DEFAULT_TOU_PRICE = np.genfromtxt("c:\\RL-Git-Projects\\BMS-DQN\\datasets\\Prices.csv", delimiter=';', skip_header=1, usecols=[-1])

In [10]:
DEFAULT_TOU_PRICE

104.0

In [23]:
day = 2
iterations = 24
time_step = 1

In [24]:
price_grid_sell = DEFAULT_TOU_PRICE[day*iterations+time_step:day*iterations+time_step+24]
price_grid_sell = np.float32(price_grid_sell/max(DEFAULT_TOU_PRICE))

In [32]:
np.array([1, 0.5])

array([1. , 0.5])

In [28]:
state = np.concatenate((np.array([1, 0.5]), price_grid_sell))

In [42]:
state.shape

(26,)

In [35]:
observation_space = spaces.Box(low=0, high=1, dtype=np.float32,
                                            shape=(26,))

In [39]:
observation_space.shape

(26,)

In [5]:
ACTIONS = [i for i in range(-80, 90, 10)]

In [6]:
ACTIONS

[-80, -70, -60, -50, -40, -30, -20, -10, 0, 10, 20, 30, 40, 50, 60, 70, 80]

In [7]:
action_space = spaces.Discrete(9)

In [8]:
action_space.n

9

In [52]:
action_space.sample()

5

# Battery settings without SOC min/max penalety

In [12]:
class Battery():
    '''simulate a simple battery here'''
    def __init__(self,capacity, max_soc, min_soc, efficiency, degradation):
        self.capacity=capacity
        self.max_soc=max_soc
        # self.initial_capacity=parameters['initial_capacity']
        self.min_soc=min_soc # 0.2
        self.degradation=degradation # degradation cost 1.2
        self.efficiency=efficiency
    def step(self,action_battery):
        energy=action_battery
        updated_capacity=max(self.min_soc,min(self.max_soc,(self.current_capacity*self.capacity+energy)/self.capacity))
        self.energy_change=(updated_capacity-self.current_capacity)*self.capacity# if charge, positive, if discharge, negative
        self.current_capacity=updated_capacity# update capacity to current codition
    def _get_cost(self,energy):# calculate the cost depends on the energy change
        cost=energy**2*self.degradation
        return cost  
    def SOC(self):
        return self.current_capacity
    def reset(self):
        self.current_capacity=np.random.uniform(0.2,0.8)
        print(f'Battery initial soc: {self.current_capacity}')


In [13]:
battery = Battery(capacity=200, max_soc=0.8, min_soc=0.2, efficiency=1, degradation=2)

In [14]:
battery.reset()

Battery initial soc: 0.6557103124154486


In [27]:
battery.step(80)

In [28]:
battery.energy_change

0.0

In [29]:
battery.current_capacity

0.8

In [10]:
from collections import namedtuple, deque
import random

In [44]:
Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))

In [12]:
memory = deque([], maxlen=5)

In [25]:
memory.append(Transition(1, 2, 3, 5, True))

In [26]:
memory

deque([Transition(state=1, action=2, reward=3, next_state=4, done=True),
       Transition(state=1, action=2, reward=3, next_state=4, done=True),
       Transition(state=1, action=2, reward=3, next_state=4, done=True),
       Transition(state=1, action=2, reward=3, next_state=4, done=True),
       Transition(state=1, action=2, reward=3, next_state=5, done=True)],
      maxlen=5)

In [49]:
memory[0].state

1

In [32]:
transitions = random.sample(memory, 2)

In [33]:
transitions

[Transition(state=1, action=2, reward=3, next_state=5, done=True),
 Transition(state=1, action=2, reward=3, next_state=4, done=True)]

In [41]:
z = zip(*transitions)

In [42]:
tuple(z)

((1, 1), (2, 2), (3, 3), (5, 4), (True, True))

In [46]:
z = Transition(*zip(*transitions))

In [47]:
z

Transition(state=(1, 1), action=(2, 2), reward=(3, 3), next_state=(5, 4), done=(True, True))

In [54]:
z.state

(1, 1)

In [6]:
action_space = spaces.Box(low=np.array([0, 0]), high=np.array([3, 1]), dtype=np.float16)

c:\Users\R00208303\Anaconda3\envs\rlcodenv\lib\site-packages\gym\spaces\box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float16
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [7]:
action_space.sample()

array([2.781, 0.797], dtype=float16)

In [4]:
observation_space = spaces.Box(
            low=0, high=1, shape=(6,), dtype=np.float16)

In [5]:
observation_space.shape[0]

6

In [54]:
a = observation_space.sample()
a

array([[0.4514 , 0.3538 , 0.03204, 0.9897 , 0.9844 , 0.6846 ],
       [0.1832 , 0.907  , 0.459  , 0.3855 , 0.7026 , 0.2764 ],
       [0.5513 , 0.743  , 0.288  , 0.9937 , 0.8643 , 0.637  ],
       [0.687  , 0.883  , 0.719  , 0.02437, 0.3032 , 0.267  ],
       [0.0717 , 0.4373 , 0.3557 , 0.8945 , 0.8047 , 0.2496 ],
       [0.5737 , 0.894  , 0.761  , 0.2559 , 0.0809 , 0.9893 ]],
      dtype=float16)

In [10]:
batch_start = np.arange(0, 6, 5)
batch_start

array([0, 5])

In [ ]:
indices = np.arange(6, dtype=np.int64)

In [2]:
ACTIONS = [[i, j] for i in range(0, 220, 20) for j in range(-80, 120, 40)]

In [3]:
ACTIONS

[[0, -80],
 [0, -40],
 [0, 0],
 [0, 40],
 [0, 80],
 [20, -80],
 [20, -40],
 [20, 0],
 [20, 40],
 [20, 80],
 [40, -80],
 [40, -40],
 [40, 0],
 [40, 40],
 [40, 80],
 [60, -80],
 [60, -40],
 [60, 0],
 [60, 40],
 [60, 80],
 [80, -80],
 [80, -40],
 [80, 0],
 [80, 40],
 [80, 80],
 [100, -80],
 [100, -40],
 [100, 0],
 [100, 40],
 [100, 80],
 [120, -80],
 [120, -40],
 [120, 0],
 [120, 40],
 [120, 80],
 [140, -80],
 [140, -40],
 [140, 0],
 [140, 40],
 [140, 80],
 [160, -80],
 [160, -40],
 [160, 0],
 [160, 40],
 [160, 80],
 [180, -80],
 [180, -40],
 [180, 0],
 [180, 40],
 [180, 80],
 [200, -80],
 [200, -40],
 [200, 0],
 [200, 40],
 [200, 80]]

In [4]:
class MicroGridEnv(gym.Env):
    def __init__(self,**kwargs):

        #self.action_space_sep = spaces.Box(low=0, high=1, dtype=np.float32,
                                       #shape=(13,))

        # Total 55 combination of actions = [(0, -80), (0, -40)...]
        self.action_space = spaces.Discrete(55)

        # Observations: loads + battery soc + res generation + price + time-step of day (any array of shape = (5,))
        self.observation_space = spaces.Box(low=-400, high=400, dtype=np.float32,
                                            shape=(5,))

In [5]:
env = MicroGridEnv()
print(env.observation_space.shape[0])
print(env.action_space.n)

5
55


In [6]:
import random
random.randint(0, actionCnt - 1)

NameError: name 'actionCnt' is not defined

In [66]:
class Battery():
    '''simulate a simple battery here'''
    def __init__(self,capacity, max_soc, min_soc, efficiency, degradation, penalty_soc):
        self.capacity = capacity # max capacity 200 kwh
        self.max_soc = max_soc # max soc 0.8
        self.min_soc = min_soc # 0.2
        self.efficiency = efficiency # charge and discharge efficiency 1.0
        self.degradation = degradation # degradation cofficient set to 0
        self.pen_soc = penalty_soc

    def step(self,battery_action):
        p_bat = battery_action
        updated_capacity = (self.current_capacity+(p_bat/self.capacity))
        #print(f'Updated capacity: {updated_capacity}')
        self.energy_change=(updated_capacity-self.current_capacity)*self.capacity# if charge -> positive, if discharge-> negative
        self.current_capacity=updated_capacity# update capacity to current codition 
        # print(f'Power charg/disch: {self.energy_change}')
        print(f'Bat current capacity: {self.current_capacity}')
        #print('---****')

    def _get_cost(self,energy):# calculate the cost depends on the energy change
        cost = (energy**2)*self.degradation
        return cost
    
    @property
    def SOC(self):
        return self.current_capacity
    
    def reset(self):
        #self.current_capacity=round(np.random.uniform(0.2,0.8), 1)  # initial capcity rounded to 1 to get 0.2, 0.3,...
        self.current_capacity=0.3      # set to 30%

In [67]:
battery = Battery(capacity=200, max_soc=0.8, min_soc=0.2, efficiency=1, degradation=0, penalty_soc=5000)

In [68]:
battery.reset()

In [69]:
print(f'Battery SOC: {battery.SOC}')
battery.step(40)
print(battery.energy_change)
print(f'Battery SOC: {battery.SOC}')

Battery SOC: 0.3
Bat current capacity: 0.5
40.0
Battery SOC: 0.5


In [4]:
class Battery():
    '''simulate a simple battery here'''
    def __init__(self,capacity, max_soc, min_soc, efficiency, degradation):
        self.capacity = capacity # max capacity 200 kwh
        self.max_soc = max_soc # max soc 0.8
        self.min_soc = min_soc # 0.2
        self.efficiency = efficiency # charge and discharge efficiency 1.0
        self.degradation = degradation # degradation cofficient set to 0

    def step(self,battery_action):
        p_bat = battery_action 
        updated_capacity = max(self.min_soc,min(self.max_soc,(self.current_capacity*self.capacity+p_bat)/self.capacity))
        self.energy_change=(updated_capacity-self.current_capacity)*self.capacity# if charge, positive, if discharge, negative
        self.current_capacity=updated_capacity# update capacity to current codition 
        print(self.energy_change)

    def _get_cost(self,energy):# calculate the cost depends on the energy change
        cost = (energy**2)*self.degradation
        return cost
    
    @property
    def SOC(self):
        return self.current_capacity
    
    def reset(self):
        self.current_capacity=round(np.random.uniform(0.2,0.8), 1)  # initial capcity rounded to 1 to get 0.2, 0.3,...


In [5]:
battery = Battery(capacity=200, max_soc=0.8, min_soc=0.2, efficiency=1, degradation=0)


In [7]:
battery.reset()

In [13]:
print(f'Battery SOC: {battery.SOC}')
battery.step(80)
print(battery.energy_change)
print(f'Battery SOC: {battery.SOC}')

Battery SOC: 0.2
0.0
0.0
Battery SOC: 0.2


In [ ]:
class Generation:
    def __init__(self, generation):
        self.power = generation


    def current_generation(self, time):
        # We consider that we have 2 sources of power a constant source and a variable source
        return self.power[time]

class Load:
    def __init__(self, base_load):
        self.base_load = base_load

    def current_load(self, time):
        return self.base_load[time]

In [ ]:
DEFAULT_POWER_GENERATED = np.genfromtxt("pv_generation_fortum.csv", delimiter=',', skip_header=0, usecols=[-1])
DEFAULT_BASE_LOAD = np.array(
    [6.658, 6.500, 6.158, 5.960, 5.883, 5.883, 6.153, 6.810, 7.780, 8.263, 8.415, 8.318, 8.350, 8.423, 8.308, 8.335, 8.485, 8.700, 9.153, 9.213, 8.865, 8.388, 7.823, 7.228])
class MicroGridEnv(gym.Env):
    def __init__(self,**kwargs):

        self.generation = Generation(kwargs.get("generation_data", DEFAULT_POWER_GENERATED))
        self.load = Load(kwargs.get('load_data', DEFAULT_BASE_LOAD))

    def crnt_gen(self, time):
         return self.generation.current_generation(time)

    def crnt_load(self):
        return self.load.current_load(23 % 24)

In [ ]:
tenv = MicroGridEnv()
tenv.crnt_gen(23)


5.16e-05

In [ ]:
DEFAULT_POWER_GENERATED = np.genfromtxt("pv_generation_fortum.csv", delimiter=',', skip_header=0, usecols=[-1])

In [ ]:
DEFAULT_POWER_GENERATED[23]

5.16e-05

In [ ]:
class Brain:
    def __init__(self, stateCnt: int, actionCnt: int, hidden_layers: Optional[List[int]] = None):
        self.stateCnt = stateCnt
        self.actionCnt = actionCnt
        self.hidden_layers = hidden_layers
        self.model = self._get_model()

    def _get_model(self):
        """
        Feed-forward network, made of linear layers with ReLU activation functions
        The number of layers, and their size is given by `hidden_layers`.
        Output layer is with linear activation as it is regression problem
        getting Q values for two actions
        """
        # assert init_method in {'default', 'xavier'}

        if self.hidden_layers is None:
            # linear model
            model = nn.Sequential(nn.Linear(self.stateCnt, self.actionCnt))

        else:
            # neural network
            # there are hidden layers in this case.
            dims = [self.stateCnt] + self.hidden_layers + [self.actionCnt]
            modules = []
            for i, dim in enumerate(dims[:-2]):
                modules.append(nn.Linear(dims[i], dims[i + 1]))
                modules.append(nn.ReLU())

            modules.append(nn.Linear(dims[-2], dims[-1]))
            model = nn.Sequential(*modules)
            # stop()

        # n_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
        # print(f'{n_parameters:,} parameters')
        return model

In [ ]:
brain = Brain(5, 55, [256, 256, 256])
brain._get_model()

Sequential(
  (0): Linear(in_features=5, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=256, bias=True)
  (3): ReLU()
  (4): Linear(in_features=256, out_features=256, bias=True)
  (5): ReLU()
  (6): Linear(in_features=256, out_features=55, bias=True)
)

In [ ]:
from keras.models import Sequential
from keras.layers import *
from keras.optimizers import *
from keras.models import *
from keras.layers import *
from keras import backend as K

In [ ]:
def _createModel(stateCnt=5, actionCnt=4):
    l_input = Input(batch_shape=(None, stateCnt))
    l_dense1 = Dense(256, activation='relu')(l_input)
    l_dense2 = Dense(256, activation='relu')(l_dense1)
    out_value = Dense(actionCnt, activation='linear')(l_dense2)
    # model = Model(inputs=l_input, outputs=[out_tcl_actions,out_price_actions,out_deficiency_actions,out_excess_actions, out_value])
    model = Model(inputs=l_input, outputs=out_value)
    model._make_predict_function()
    opt = RMSprop(lr=0.00025)
    model.compile(loss='mse', optimizer=opt)
    return model

In [ ]:
_createModel().summary()

Model: "model_6"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_7 (InputLayer)         [(None, 5)]               0         
_________________________________________________________________
dense_18 (Dense)             (None, 256)               1536      
_________________________________________________________________
dense_19 (Dense)             (None, 256)               65792     
_________________________________________________________________
dense_20 (Dense)             (None, 4)                 1028      
Total params: 68,356
Trainable params: 68,356
Non-trainable params: 0
_________________________________________________________________


In [ ]:
# Run only if Kernel is dying due to matplotlib plt command
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

# 2. Types of Spaces

In [ ]:
action_space = Discrete(4)
action_space.n

4

In [ ]:
observation_space = Box(low=-100, high=100, dtype=np.float32,
                                            shape=(5,))
observation_space.shape[0]

5

In [ ]:
state = np.array([1, 2, 3, 4, 5])
state

array([1, 2, 3, 4, 5])

In [ ]:
Box(0,2,shape=(3,3)).sample()

array([[1.6432045 , 0.16452263, 0.02527925],
       [0.33831087, 1.5018448 , 0.35238257],
       [1.8728477 , 0.6129641 , 1.6607267 ]], dtype=float32)

In [ ]:
Tuple((Discrete(3), Box(0,100, shape=(1,)))).sample()

(0, array([38.00223], dtype=float32))

In [ ]:
Box(low=np.array([0, 0]), high=np.array([100, 100])).sample()


array([84.12775 , 24.751968], dtype=float32)

In [ ]:
Dict({'height':Discrete(2), "speed":Box(0,100, shape=(1,))}).sample()

OrderedDict([('height', 0), ('speed', array([18.216503], dtype=float32))])

In [ ]:
MultiBinary(4).sample()

array([0, 1, 0, 1], dtype=int8)

In [ ]:
ACTIONS = [[i, j] for i in range(2) for j in range(2)]

In [ ]:
ACTIONS

[[0, 0], [0, 1], [1, 0], [1, 1]]

In [ ]:
action_space = Discrete(4)  # 0 - 3
actionCnt =action_space.n
actionCnt

4

In [ ]:
action = random.randint(0, actionCnt - 1)

In [ ]:
if type(action) is not list:
    action = ACTIONS[action]

In [ ]:
action 

[0, 1]

# 3. Building an Environment
- Build an agent to give us the best shower possible
- Randomly Temperature (it is going to fluctuate because people are in the building it is going to randomly up and down)
- 37 and 39 dgrees (our optimal temperature between 37 -  39)
- So we want to train an agent to automatically respond to changes in temperature and get it within 37 and 39

In [ ]:
class ShowerEnv(Env):
    def __init__(self):
        # Actions we can take, turn the tap down, stay, up
        self.action_space = spaces.Discrete(80)
        # Temperature array
        # self.observation_space = Box(low=0, high=100, shape=(1,)) an other way of initiliasing a box
        self.observation_space = Box(low=np.array([0]), high=np.array([100]))
        # Set start temp (initial state, shower is gonna start at 38 -+ 3)
        self.state = 38 + random.randint(-3,3)
        # Set shower length, representing the episode length of 60 sec
        self.shower_length = 60
        
    def step(self, action):
        # Apply action (applying the impact of our action on our state)
        # 0 -1 = -1 temperature (dec temp by 1 deg)
        # 1 -1 = 0  no change
        # 2 -1 = 1 temperature (inc temp by 1 deg)
        self.state += action -1 
        # Reduce shower length by 1 second
        self.shower_length -= 1 
        
        # Calculate reward, it is +1 if state is between/equal 37 and 39 else -1
        if self.state >=37 and self.state <=39: 
            reward =1 
        else: 
            reward = -1 
        
        # Check if shower is done (episode is done)
        if self.shower_length <= 0: 
            done = True
        else:
            done = False
        
        # Apply temperature noise
        #self.state += random.randint(-1,1)
        # Set placeholder for info
        info = {}
        
        # Return step information
        return self.state, reward, done, info
    
    def render(self):
        # implement viz
        pass
    
    def reset(self):
        # Reset shower temperature
        self.state = np.array([38 + random.randint(-3,3)]).astype(float)
        # Reset shower time
        self.shower_length = 60 
        return self.state
    

NameError: name 'Env' is not defined

In [ ]:
env = ShowerEnv()

C:\Users\R00223013\Anaconda3\envs\rlcodenv\lib\site-packages\gym\spaces\box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [ ]:
env.observation_space.sample()

array([66.16909], dtype=float32)

In [ ]:
env.action_space.sample()

2

# 4. Test Environment

In [ ]:
episodes = 5
for episode in range(1, episodes+1):
    obs = env.reset()       # Resets the environment  
    done = False             
    score = 0 
    
    while not done:
        env.render()       # View or pop-up environment but it is not required anymore as render_mode == 'human' is pas
        action = env.action_space.sample()       # Random actions are being taken
        # 
        obs, reward, done, info = env.step(action)    # apply an action to the environment to return obs, reward,..
        score+=reward
    print('Episode:{} Score:{}'.format(episode, score))
env.close()      # close down the render frame

Episode:1 Score:-60
Episode:2 Score:-46
Episode:3 Score:38
Episode:4 Score:-2
Episode:5 Score:-42


# 5. Train Model

In [ ]:
log_path = os.path.join('training', 'logs')
model = PPO("MlpPolicy", env, verbose=1, tensorboard_log=log_path)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [ ]:
model.learn(total_timesteps=400000)

Logging to training\logs\PPO_11
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 60       |
|    ep_rew_mean     | -29.2    |
| time/              |          |
|    fps             | 2280     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60          |
|    ep_rew_mean          | -29.6       |
| time/                   |             |
|    fps                  | 1601        |
|    iterations           | 2           |
|    time_elapsed         | 2           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011149089 |
|    clip_fraction        | 0.0562      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | -0.000149   

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60          |
|    ep_rew_mean          | -17.7       |
| time/                   |             |
|    fps                  | 1282        |
|    iterations           | 11          |
|    time_elapsed         | 17          |
|    total_timesteps      | 22528       |
| train/                  |             |
|    approx_kl            | 0.012247643 |
|    clip_fraction        | 0.0488      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.05       |
|    explained_variance   | -0.00015    |
|    learning_rate        | 0.0003      |
|    loss                 | 37.7        |
|    n_updates            | 100         |
|    policy_gradient_loss | -0.00398    |
|    value_loss           | 69.4        |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60    

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60          |
|    ep_rew_mean          | -3.08       |
| time/                   |             |
|    fps                  | 1262        |
|    iterations           | 21          |
|    time_elapsed         | 34          |
|    total_timesteps      | 43008       |
| train/                  |             |
|    approx_kl            | 0.003481444 |
|    clip_fraction        | 0.0315      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.772      |
|    explained_variance   | 0.0047      |
|    learning_rate        | 0.0003      |
|    loss                 | 45.7        |
|    n_updates            | 200         |
|    policy_gradient_loss | -0.000804   |
|    value_loss           | 76.5        |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60    

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60          |
|    ep_rew_mean          | 35.4        |
| time/                   |             |
|    fps                  | 1256        |
|    iterations           | 31          |
|    time_elapsed         | 50          |
|    total_timesteps      | 63488       |
| train/                  |             |
|    approx_kl            | 0.011136094 |
|    clip_fraction        | 0.145       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.83       |
|    explained_variance   | -0.000112   |
|    learning_rate        | 0.0003      |
|    loss                 | 30.3        |
|    n_updates            | 300         |
|    policy_gradient_loss | -0.00628    |
|    value_loss           | 59.5        |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60    

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 60           |
|    ep_rew_mean          | 47.9         |
| time/                   |              |
|    fps                  | 1255         |
|    iterations           | 41           |
|    time_elapsed         | 66           |
|    total_timesteps      | 83968        |
| train/                  |              |
|    approx_kl            | 0.0064330497 |
|    clip_fraction        | 0.164        |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.792       |
|    explained_variance   | 7.21e-06     |
|    learning_rate        | 0.0003       |
|    loss                 | 43.5         |
|    n_updates            | 400          |
|    policy_gradient_loss | 0.00498      |
|    value_loss           | 82.5         |
------------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_m

...

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 60           |
|    ep_rew_mean          | 52.6         |
| time/                   |              |
|    fps                  | 1250         |
|    iterations           | 151          |
|    time_elapsed         | 247          |
|    total_timesteps      | 309248       |
| train/                  |              |
|    approx_kl            | 0.0028026542 |
|    clip_fraction        | 0.0501       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.247       |
|    explained_variance   | -0.00192     |
|    learning_rate        | 0.0003       |
|    loss                 | 60.2         |
|    n_updates            | 1500         |
|    policy_gradient_loss | 0.00208      |
|    value_loss           | 107          |
------------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_m

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60          |
|    ep_rew_mean          | 56.5        |
| time/                   |             |
|    fps                  | 1251        |
|    iterations           | 161         |
|    time_elapsed         | 263         |
|    total_timesteps      | 329728      |
| train/                  |             |
|    approx_kl            | 0.005210144 |
|    clip_fraction        | 0.0566      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.376      |
|    explained_variance   | -2.96e-05   |
|    learning_rate        | 0.0003      |
|    loss                 | 64.4        |
|    n_updates            | 1600        |
|    policy_gradient_loss | 0.00499     |
|    value_loss           | 117         |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60    

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60          |
|    ep_rew_mean          | 55.9        |
| time/                   |             |
|    fps                  | 1252        |
|    iterations           | 171         |
|    time_elapsed         | 279         |
|    total_timesteps      | 350208      |
| train/                  |             |
|    approx_kl            | 0.017870214 |
|    clip_fraction        | 0.108       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.521      |
|    explained_variance   | -7.63e-06   |
|    learning_rate        | 0.0003      |
|    loss                 | 56.3        |
|    n_updates            | 1700        |
|    policy_gradient_loss | 0.00951     |
|    value_loss           | 118         |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60    

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60          |
|    ep_rew_mean          | 57.3        |
| time/                   |             |
|    fps                  | 1253        |
|    iterations           | 181         |
|    time_elapsed         | 295         |
|    total_timesteps      | 370688      |
| train/                  |             |
|    approx_kl            | 0.008330503 |
|    clip_fraction        | 0.0905      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.404      |
|    explained_variance   | 0.00289     |
|    learning_rate        | 0.0003      |
|    loss                 | 51.1        |
|    n_updates            | 1800        |
|    policy_gradient_loss | 0.00774     |
|    value_loss           | 116         |
-----------------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 60      

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60          |
|    ep_rew_mean          | 34.5        |
| time/                   |             |
|    fps                  | 1253        |
|    iterations           | 191         |
|    time_elapsed         | 311         |
|    total_timesteps      | 391168      |
| train/                  |             |
|    approx_kl            | 0.040321298 |
|    clip_fraction        | 0.17        |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.507      |
|    explained_variance   | -0.000363   |
|    learning_rate        | 0.0003      |
|    loss                 | 37.9        |
|    n_updates            | 1900        |
|    policy_gradient_loss | -0.0031     |
|    value_loss           | 73.9        |
-----------------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 60  

# 6. Save Model

In [ ]:
shower_path = os.path.join('training', 'savedmodels', 'Shower_Model_PPO')

In [ ]:
model.save(shower_path)

In [ ]:
del model

In [ ]:
model = PPO.load(shower_path, env)

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


C:\Users\R00223013\Anaconda3\envs\rlcodenv\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


In [ ]:
evaluate_policy(model, env, n_eval_episodes=10, render=True)

C:\Users\R00223013\Anaconda3\envs\rlcodenv\lib\site-packages\stable_baselines3\common\evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(
C:\Users\R00223013\Anaconda3\envs\rlcodenv\lib\site-packages\stable_baselines3\common\vec_env\base_vec_env.py:234: UserWarning: You tried to call render() but no `render_mode` was passed to the env constructor.
  warnings.warn("You tried to call render() but no `render_mode` was passed to the env constructor.")


(35.8, 47.90365330535866)

In [ ]:
# mean episode reward of 35.8 with std of 47.9